### Imports and Environment Setup

In [1]:
import os
import cv2
import time
import numpy as np
import pandas as pd
from kagglehub import dataset_download
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# MLflow imports (commented out as requested)
# import mlflow
# import mlflow.sklearn
# from mlflow.models.signature import infer_signature

In [2]:
kaggle_path = dataset_download("datajameson/oxford-17-flowers-dataset")

### Data Loading and Feature Extraction Function

In [3]:
def load_data(data_path, image_size=(128, 128)):
    """Load images, extract features, and encode labels."""
    X = []
    y = []
    
    print("Loading images... Please wait.")
    for label in os.listdir(data_path):
        dir_path = os.path.join(data_path, label)
        
        if not os.path.isdir(dir_path):
            continue

        for img_name in os.listdir(dir_path):
            img_path = os.path.join(dir_path, img_name)
            image = cv2.imread(img_path)

            if image is None:
                print(f"Could not read: {img_path}")
                continue

            image = cv2.resize(image, image_size)

            # Histogram features
            hist = cv2.calcHist([image], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
            cv2.normalize(hist, hist)
            histogram_features = hist.flatten()

            # Hu Moments features
            image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            hu_features = cv2.HuMoments(cv2.moments(image_gray)).flatten()
            hu_features = -np.sign(hu_features) * np.log10(np.abs(hu_features) + 1e-10)

            current_features = np.hstack([histogram_features, hu_features])
            X.append(current_features)
            y.append(label)

    X = np.array(X)
    y = np.array(y)
    print(f"Extraction complete! Extracted {X.shape[1]} features per image for {X.shape[0]} images.")
    
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    return X, y_encoded

### Modeling and Experiment Tracking Functions

In [4]:
def get_preprocessor():
    """Create preprocessing step."""
    return StandardScaler()

In [5]:
def train_model(X_train, y_train, model, param_grid):
    """Train a model using GridSearchCV and a scaling pipeline."""
    preprocessor = get_preprocessor()
    
    model_pipeline = Pipeline(steps=[
        ('scaler', preprocessor),
        ('model', model)
    ])
    
    grid_search = GridSearchCV(
        model_pipeline,
        param_grid,
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X_train, y_train)
    return grid_search

In [6]:
def evaluate_model(model, X_test, y_test):
    """Evaluate model and return metrics."""
    test_preds = model.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, test_preds)
    }

In [7]:
def run_experiment(X_train, X_test, y_train, y_test, model_name, model, param_grid):
    """Run an experiment with MLflow tracking (commented out)."""
    
    # MLflow implementation
    # with mlflow.start_run(run_name=model_name):
    #     mlflow.log_params({"model_name": model_name})
        
    start_time = time.time()
    fitted_grid = train_model(X_train, y_train, model, param_grid)
    end_time = time.time()

    best_model = fitted_grid.best_estimator_
    metrics = evaluate_model(best_model, X_test, y_test)

    #     mlflow.log_metric("best_cv_accuracy", fitted_grid.best_score_)
    #     mlflow.log_metric("test_accuracy", metrics['accuracy'])
    #     
    #     signature = infer_signature(X_train, best_model.predict(X_train))
    #     mlflow.sklearn.log_model(best_model, "model", signature=signature)
    
    return {
        'Algorithm': model_name,
        'Best CV Accuracy': round(fitted_grid.best_score_, 4),
        'Final Test Accuracy': round(metrics['accuracy'], 4),
        'Time (seconds)': round(end_time - start_time, 2),
        'Best Hyperparameters': fitted_grid.best_params_
    }

### Execution Pipeline

In [8]:
# mlflow.set_experiment("Image Classification Experiment")

# IMPORTANT: Set this to your actual directory!
data_path = os.path.join(kaggle_path, 'Oxford 17 Flowers')

# 1. Load Data
X, y = load_data(data_path)

# 2. Split Data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=132, stratify=y
)

# 3. Define Models and Parameter Grids
experiments = [
    {
        'model_name': 'RBF Kernel SVM',
        'model': SVC(random_state=121),
        'params': {
            'model__C': [0.1, 1.0, 10.0, 100.0],
            'model__gamma': ['scale', 'auto', 0.01, 0.1]
        }
    },
    {
        'model_name': 'Logistic Regression',
        'model': LogisticRegression(random_state=121, max_iter=2000),
        'params': {
            'model__C': [0.01, 0.1, 1.0, 10.0],
            'model__solver': ['lbfgs', 'liblinear']
        }
    },
    {
        'model_name': 'Random Forest',
        'model': RandomForestClassifier(random_state=121),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [None, 15, 30],
            'model__max_features': ['sqrt', 'log2']
        }
    },
    # {
    #     'model_name': 'Hist Gradient Boosting',
    #     'model': HistGradientBoostingClassifier(random_state=121),
    #     'params': {
    #         'model__learning_rate': [0.01, 0.1],
    #         'model__max_iter': [100, 200]
    #     }
    # },
    {
        'model_name': 'k-Nearest Neighbors',
        'model': KNeighborsClassifier(),
        'params': {
            'model__n_neighbors': [3, 5, 7, 11],
            'model__weights': ['uniform', 'distance']
        }
    }
]

# 4. Run Experiments
results = []
for exp in experiments:
    print(f"Training and tuning {exp['model_name']}...")
    result = run_experiment(
        X_train, X_val, y_train, y_val,
        exp['model_name'], exp['model'], exp['params']
    )
    results.append(result)
    print(f"{exp['model_name']} completed in {result['Time (seconds)']}s.")

# 5. Build and Display Summary Table
results_df = pd.DataFrame(results).sort_values(by='Final Test Accuracy', ascending=False).reset_index(drop=True)

print("\n" + "="*50)
print("FINAL MODEL COMPARISON")
print("="*50)
display(results_df)

Loading images... Please wait.
Extraction complete! Extracted 519 features per image for 1360 images.
Training and tuning RBF Kernel SVM...
RBF Kernel SVM completed in 7.48s.
Training and tuning Logistic Regression...
Logistic Regression completed in 12.01s.
Training and tuning Random Forest...
Random Forest completed in 11.87s.
Training and tuning k-Nearest Neighbors...
k-Nearest Neighbors completed in 0.36s.

FINAL MODEL COMPARISON


,Algorithm,Best CV Accuracy,Final Test Accuracy,Time (seconds),Best Hyperparameters
0,Random Forest,0.6578,0.6765,11.87,"{'model__max_depth': None, 'model__max_feature..."
1,RBF Kernel SVM,0.4784,0.4765,7.48,"{'model__C': 10.0, 'model__gamma': 'auto'}"
2,Logistic Regression,0.4971,0.4735,12.01,"{'model__C': 0.1, 'model__solver': 'lbfgs'}"
3,k-Nearest Neighbors,0.3931,0.4118,0.36,"{'model__n_neighbors': 5, 'model__weights': 'd..."
